In [37]:
from airports.airportLookup import AirportLookup
airport_lookup = AirportLookup()
import polars as pl
def add_airport_coords(df: pl.DataFrame, prefix: str) -> pl.DataFrame:
    airport_col = f"{prefix}_airport_ident"
    lat_col = f"{prefix}_airport_lat"
    lon_col = f"{prefix}_airport_lon"

    def get_coords(airport_ident: str | None) -> dict[str, float | None]:
        coords = airport_lookup.get_airport_coordinates(airport_ident)

        if coords is None:
            return {lat_col: None, lon_col: None}

        lat, lon = coords
        return {lat_col: lat, lon_col: lon}

    return (
        df.with_columns(
            pl.col(airport_col)
            .map_elements(
                get_coords,
                return_dtype=pl.Struct({
                    lat_col: pl.Float64,
                    lon_col: pl.Float64,
                }),
            )
            .alias("_coords")
        )
        .unnest("_coords")
    )


In [51]:
from data_engineering.flights.flight_type import flights_df_to_flights, get_flights
from datetime import date
df = get_flights(date(2025,5, 18), date(2025,5,24))
df = df.filter(pl.col("ladd") | pl.col("pia"))
df = df.filter(pl.col("category") != "A7")
df = add_airport_coords(df, "takeoff")
df = add_airport_coords(df, "landing")

In [ ]:
from airports.airportLookup import Airport


def select_bigger_airport(airport_0: Airport, airport_1: Airport):
    if airport_0.type > airport_1.type:
        return airport_0
    return airport_1

from utils import haversine
count = 0
new_flights = []
ladd_icaos = set()
def process_icao_group(group):
    flights = flights_df_to_flights(group)
    previous_flight = None
    if len(flights) > 7:
        return # training plane
    for flight in flights:
        if previous_flight is not None:
            if previous_flight.landing_airport_ident != flight.takeoff_airport_ident:
                distance = haversine(
                    previous_flight.landing_airport_lat,
                    previous_flight.landing_airport_lon,
                    flight.takeoff_airport_lat,
                    flight.takeoff_airport_lon,
                )
                takeoff_airport = airport_lookup.get_Airport_from_airport_ident(flight.takeoff_airport_ident)
                landing_airport = airport_lookup.get_Airport_from_airport_ident(previous_flight.landing_airport_ident)
                if takeoff_airport is None or landing_airport is None:
                    raise Exception("airports are None")
                # A good test is if it can identify which icaos are pia. 
                if flight.callsign == previous_flight.callsign and distance <1000: # careful this includes callsigns that start with "N*"
                    airport = select_bigger_airport(takeoff_airport,landing_airport)
                    flight.takeoff_airport_ident = airport.ident
                    previous_flight.landing_airport_ident = airport.ident
                elif distance < 100:
                    airport = select_bigger_airport(takeoff_airport, landing_airport)
                    flight.takeoff_airport_ident = airport.ident
                    previous_flight.landing_airport_ident = airport.ident
                else: # cranking the distance is good.
                    if airport_lookup.airport_ident_is_in_us_faa_airspace(previous_flight.landing_airport_ident) \
                        and airport_lookup.airport_ident_is_in_us_faa_airspace(flight.takeoff_airport_ident):
                        ladd_icaos.add(flight.icao)
                    else:
                        pass # international                    

        previous_flight = flight
    new_flights.extend(flights)

In [76]:
for icao_key, group in df.group_by("icao"):
    process_icao_group(group)

350.88746841143825


In [ ]:
display(df.get_column("icao").unique())

icao
str
"""a6a068"""
"""adbdb9"""
"""a98ac8"""
"""e499e4"""
"""a6338b"""
…
"""a84f35"""
"""502d94"""
"""a55ded"""


18


In [88]:
ladd_icaos

{'4d2039',
 'a0069d',
 'a01c3a',
 'a0b303',
 'a12c04',
 'a1addc',
 'a21397',
 'a2c4e0',
 'a4c5b0',
 'a5d02e',
 'a793b2',
 'a79556',
 'a79b1c',
 'a8d5dd',
 'aa7032',
 'aa80ca',
 'aabad0',
 'ac959a'}

In [90]:
from data_engineering.flights.flight_type import flights_to_flights_df


df_new = flights_to_flights_df(new_flights)
df_new.filter(pl.col("icao") == "a0069d")

icao,callsign,registration,takeoff_time,takeoff_airport_ident,landing_time,landing_airport_ident,pia,ladd,military,interesting,aircraft_type,owner,aircraft_description,category,takeoff_airport_lat,takeoff_airport_lon,landing_airport_lat,landing_airport_lon
str,str,str,datetime[ms],str,datetime[ms],str,bool,bool,bool,bool,str,str,str,str,f64,f64,f64,f64
"""a0069d""","""KFB100""","""N100VP""",2025-05-18 16:39:20.060,"""KABQ""",2025-05-18 17:56:48.090,"""US-4717""",false,true,false,false,"""C560""","""VCS ENCORE LLC""","""CESSNA 560 Citation Ultra""","""A2""",35.039976,-106.608925,33.82033,-114.86336
"""a0069d""","""N100VP""","""N100VP""",2025-05-18 19:23:43.690,"""KPSP""",2025-05-18 19:40:38.960,"""KSNA""",false,true,false,false,"""C560""","""VCS ENCORE LLC""","""CESSNA 560 Citation Ultra""","""A2""",33.8297,-116.507004,33.675701,-117.867996


In [78]:
set(df.filter(pl.col("pia")).get_column("icao").unique()) & ladd_icaos

{'a1addc'}

In [70]:
df.filter(pl.col("icao") == "a03b3f").sort("takeoff_time")

icao,callsign,registration,takeoff_time,takeoff_airport_ident,landing_time,landing_airport_ident,pia,ladd,military,interesting,aircraft_type,owner,aircraft_description,category,takeoff_airport_lat,takeoff_airport_lon,landing_airport_lat,landing_airport_lon
str,str,str,datetime[ms],str,datetime[ms],str,bool,bool,bool,bool,str,str,str,str,f64,f64,f64,f64
"""a03b3f""","""FFL1480""",null,2025-05-18 20:17:12.320,"""KMYF""",2025-05-18 21:30:39.330,"""SZP""",true,false,false,false,null,null,null,"""A1""",32.8157,-117.139999,34.347198,-119.060997
"""a03b3f""","""FFL1480""",null,2025-05-18 21:57:42.850,"""SZP""",2025-05-18 22:22:16.120,"""KWHP""",true,false,false,false,null,null,null,"""A1""",34.347198,-119.060997,34.2593,-118.413002
"""a03b3f""","""FFL1480""",null,2025-05-19 22:37:29.760,"""US-0462""",2025-05-19 23:18:26.830,"""KHWD""",true,false,false,false,null,null,null,"""A1""",36.706512,-120.860132,37.659199,-122.122002
"""a03b3f""","""FFL1480""",null,2025-05-20 06:35:33.930,"""KHWD""",2025-05-20 08:14:53.460,"""KWHP""",true,false,false,false,null,null,null,"""A1""",37.659199,-122.122002,34.2593,-118.413002


In [ ]:
# use callsign and then check if there's a big jump then there's really a problem
# !!!! This is how I detemrine which icaos are using pia which ones have a jump to not. I can probably elmianate tons of icaos from before the solve. 